### Table Maintenance

This notebook keeps the silver table performant over time. It should run on a schedule, separately from the main silver pipeline.

* `OPTIMIZE` compacts many small Parquet files into fewer, larger files, which makes future reads faster.
* `VACUUM` removes old, no-longer-referenced data files (for example, files left over from previous versions of the table after updates or deletes), freeing up storage.
* All operations are wrapped in `try/except` with logging, so that if maintenance fails, the job fails visibly (`raise`) instead of silently succeeding with an error hidden in the logs.

In [0]:
%run ./00_config

In [0]:
import logging 

logger = logging.getLogger("table_maintenance")
logger.setLevel(logging.INFO)

table_to_maintain = [target_silver_table]

for table in table_to_maintain:
    try:

        optimize_result = spark.sql(f"OPTIMIZE {table}")
        stats = optimize_result.collect()[0].asDict()
        logger.info(f"OPTIMIZE completed for {table}: {stats}")

        spark.sql(f"VACUUM {table}")
        logger.info(f"VACUUM completed for {table}")

    except Exception as e:
        logger.error(f"Error maintaining table {table}: {e}")
        raise 